# Reaction Video Editor — Colab Version

**What this is:** A Python pipeline combining both repo variants (root original + `diffrent variant`) for processing reaction videos inside Google Colab.

**What it does:**
- Splits 3840×1080 OBS side-by-side recordings into camera / content halves.
- Composes reaction layout (camera top-left rounded, content bottom-right rounded, blurred background 50%/op 40%).
- Mixes 2-track audio with compressor / limiter / sidechain ducking.
- Auto-cuts silent/repeated reaction segments for seamless flow.
- Transcribes intro/outro via Whisper to fix pauses/repeats.
- Applies face retouch (smooth, teeth, nose, eyes) with MediaPipe tracking.
- Exports MP4 and WebM directly to Google Drive.

**Where computing happens:** Inside this Colab VM (CPU; enable GPU runtime if you want). Large 3 GB files stream through ffmpeg.

**WebM to YouTube:** Yes — YouTube fully accepts VP9/WebM. We export both MP4 and WebM.

**Drive input/output:** Mount with `drive.mount()`, point `ReactionVideoProcessor` to your video, set `output_dir` to Drive.

In [ ]:
# 1) Install dependencies (run once per session)
!pip install -q numpy opencv-python mediapipe openai-whisper ffmpeg-python moviepy pydub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 13.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.0/248.0 MB 5.1 MB/s eta 0:00:00


## 2) Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3) Import processor and configure paths

In [ ]:
import sys, os
sys.path.insert(0, '/content/VideoEditorTool/colab_version')
from video_processor import ReactionVideoProcessor

input_video = '/content/drive/MyDrive/raw/recording_3840.mp4'   # <-- change
output_dir  = '/content/drive/MyDrive/reaction_output'        # <-- change

proc = ReactionVideoProcessor(input_video, output_dir=output_dir)

## 4) Patreon version — full uncut + intro/outro cleaned
- Intro / outro kept full camera (safe, no copyrighted content).
- Reaction composed with preset layout.
- Whispers intro to detect pauses / repeats for manual edit.

In [ ]:
proc.run_patron_version(
    intro_range=(12, 55),
    outro_range=(1260, 1300),
    preset='diagonal',
    retouch=False,
    fix_intro=True
)

## 5) YouTube version — reaction with alterations + auto-cuts
- Middle reaction gets silence/repeat cuts (auto_cut=True).
- Camera retouched (retouch=True) — tracking rebuilds mask every frame.
- Custom cuts can be passed: custom_cuts=[(t1,t2), ...]

In [ ]:
proc.run_youtube_version(
    preset='diagonal',
    auto_cut=True,
    retouch=True,
    intro_range=(12, 55),
    outro_range=(1260, 1300)
)

## 6) Check output files in Drive
Files written to `/content/drive/MyDrive/reaction_output/` (or your `output_dir`):
- `patreon_final.mp4` / `.webm`
- `youtube_final.mp4` / `.webm`
- `intro_transcript.json` (Whisper result)

List them with:
```python
!ls -lh /content/drive/MyDrive/reaction_output/
```

In [ ]:
!ls -lh /content/drive/MyDrive/reaction_output/ 2>/dev/null || echo 'Output folder not found yet — run pipeline above first.'

---
**Notes**
- If session dies, re-run cells 1→3; outputs in Drive survive.
- Very long videos: split at source (e.g., 0-10 min chunks) and concatenate with `ffmpeg -i p1 -i p2 -filter_complex concat`.
- Face retouch is CPU-heavy; for 20 min videos it may take 10–20 min. Skip retouch if you prefer CapCut for that step.